# 对照实验 · 通用笔记本（Colab）

用途：跑横向对比模型，填充论文对比表。**只改第 2 格的两个变量**：

| 实验 | MODEL | BATCH |
|---|---|---|
| YOLOv8n | `'yolov8n.pt'` | 16 |
| RT-DETR-l | `'rtdetr-l.pt'` | 8 |
| YOLOv12n | `'yolo12n.pt'` | 16 |

每跑完一个模型，下载 results.zip 并修改 EXPERIMENT_LOG。操作同前：T4 → 全部运行。

In [ ]:
# ===== 每次运行只改这两个变量 =====
MODEL = 'rtdetr-l.pt'   # 模型权重名
BATCH = 8               # yolo 系用 16，rtdetr 用 8
NAME = MODEL.replace('.pt', '')
print(f'本次实验: {NAME}, batch={BATCH}')

In [ ]:
!nvidia-smi

In [ ]:
# 下载数据集（同前）
API_KEY = 'YOUR_ROBOFLOW_API_KEY'

import json, glob, zipfile, urllib.request, subprocess

meta = json.load(urllib.request.urlopen(
    f'https://api.roboflow.com/km-sd0ce/pig-behavior-wlvku/1/yolov8?api_key={API_KEY}'))
subprocess.run(['curl', '-sL', '-o', '/content/dataset.zip', meta['export']['link']], check=True)
with zipfile.ZipFile('/content/dataset.zip') as z:
    z.extractall('/content/dataset')
DATA_YAML = glob.glob('/content/dataset/**/data.yaml', recursive=True)[0]
print('data.yaml:', DATA_YAML)

In [ ]:
!pip install -q ultralytics
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
# 训练（与基线同协议：100 轮 / 640）
from ultralytics import YOLO, RTDETR

model = RTDETR(MODEL) if 'rtdetr' in MODEL else YOLO(MODEL)
model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=BATCH,
            device=0, project='/content/results', name=NAME)

In [ ]:
# 评估 + 保存指标
import json

metrics = model.val()
summary = {
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50-95': round(float(metrics.box.map), 4),
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
}
import os
os.makedirs(f'/content/results/{NAME}', exist_ok=True)
with open(f'/content/results/{NAME}/metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(summary)
print('对照基线(yolo11n): mAP50=0.5706  mAP50-95=0.4169')

In [ ]:
# 打包下载
import shutil
from google.colab import files

shutil.make_archive('/content/results', 'zip', '/content/results')
files.download('/content/results.zip')